# LSTM vs Transformer — CS 110 Final Project

## What this is

So for my CS 110 final I wanted to actually *show* why Transformers replaced LSTMs instead of just writing about it. The whole argument in my paper is that LSTMs were slow because they process text one word at a time (sequential), while Transformers read everything at once (parallel) — which makes them way better at using GPU hardware efficiently.

To test this I built two tiny neural networks — one LSTM and one Transformer — and trained both of them on the exact same task: predicting the next character in a Shakespeare text. Same data, same number of epochs, same everything. Then I measured how fast each one trained and how accurate they got.

I'm not a Python expert at all, this is an intro CS class, so I kept it pretty simple. I used PyTorch because it has both architectures built in — I didn't have to build either model from scratch, just configure them. The LSTM used `nn.LSTM` and the Transformer used `nn.TransformerEncoder`, both from PyTorch's standard library. I set them up to have roughly similar sizes so the comparison would be fair.

## What the results showed

Honestly the results were kind of what I expected but still cool to see actually happen. Both models learned the task (accuracy went up, loss went down over 15 epochs), but the Transformer was noticeably faster per epoch. On a CPU the difference is modest, but the point is that on real GPU hardware at scale — like the clusters that train GPT — that speed gap becomes massive because GPU cores can all work at the same time with a Transformer, while an LSTM makes most of them sit idle waiting. That's the whole reason modern AI took off when it did.

The charts at the bottom show loss, accuracy, and time per epoch side by side. The summary cell at the end prints the actual numbers.

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'torch', 'matplotlib', 'numpy'], capture_output=True)
print('Libraries ready!')

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

print('PyTorch version:', torch.__version__)
print('All imports successful!')

In [ ]:
text = """
To be or not to be that is the question whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune or to take arms against a sea of troubles
and by opposing end them to die to sleep no more and by a sleep to say we end
the heartache and the thousand natural shocks that flesh is heir to tis a consummation
devoutly to be wished to die to sleep to sleep perchance to dream ay there is the rub
for in that sleep of death what dreams may come when we have shuffled off this mortal coil
must give us pause there is the respect that makes calamity of so long life
for who would bear the whips and scorns of time the oppressors wrong the proud mans contumely
the pangs of despised love the laws delay the insolence of office and the spurns
that patient merit of the unworthy takes when he himself might his quietus make
with a bare bodkin who would fardels bear to grunt and sweat under a weary life
but that the dread of something after death the undiscovered country from whose bourn
no traveller returns puzzles the will and makes us rather bear those ills we have
than fly to others that we know not of thus conscience does make cowards of us all
""".strip().lower()

chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
encoded = torch.tensor([char2idx[c] for c in text], dtype=torch.long)

print(f'Text length: {len(text)} characters')
print(f'Vocabulary size: {vocab_size} unique characters')

In [ ]:
SEQ_LEN = 40
BATCH_SIZE = 32

def make_batches(encoded, seq_len, batch_size):
    sequences, targets = [], []
    for i in range(0, len(encoded) - seq_len - 1, 2):
        sequences.append(encoded[i:i+seq_len])
        targets.append(encoded[i+1:i+seq_len+1])
    X = torch.stack(sequences)
    Y = torch.stack(targets)
    idx = torch.randperm(len(X))
    X, Y = X[idx], Y[idx]
    n_batches = len(X) // batch_size
    X = X[:n_batches*batch_size].view(n_batches, batch_size, seq_len)
    Y = Y[:n_batches*batch_size].view(n_batches, batch_size, seq_len)
    return X, Y

X, Y = make_batches(encoded, SEQ_LEN, BATCH_SIZE)
print(f'Input shape: {X.shape}')
print('Ready to train!')

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out)

lstm_model = LSTMModel(vocab_size)
print(f'LSTM — {sum(p.numel() for p in lstm_model.parameters()):,} parameters')

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2, seq_len=40):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=128, batch_first=True, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        positions = torch.arange(x.size(1)).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1))
        out = self.transformer(x, mask=mask, is_causal=True)
        return self.fc(out)

transformer_model = TransformerModel(vocab_size)
print(f'Transformer — {sum(p.numel() for p in transformer_model.parameters()):,} parameters')

In [ ]:
def train_model(model, X, Y, epochs=15, lr=0.003):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses, accuracies, epoch_times = [], [], []

    for epoch in range(epochs):
        model.train()
        total_loss, total_correct, total_tokens = 0, 0, 0
        start = time.time()

        for batch_idx in range(len(X)):
            xb = X[batch_idx]
            yb = Y[batch_idx]
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            preds = logits.argmax(dim=-1)
            total_correct += (preds == yb).sum().item()
            total_tokens += yb.numel()

        epoch_time = time.time() - start
        avg_loss = total_loss / len(X)
        accuracy = total_correct / total_tokens * 100
        losses.append(avg_loss)
        accuracies.append(accuracy)
        epoch_times.append(epoch_time)
        print(f'  Epoch {epoch+1:2d}/{epochs} — Loss: {avg_loss:.4f}  Accuracy: {accuracy:.1f}%  Time: {epoch_time:.2f}s')

    return losses, accuracies, epoch_times

In [ ]:
print('Training LSTM...')
print('=' * 55)
lstm_losses, lstm_acc, lstm_times = train_model(lstm_model, X, Y)
print(f'\nTotal: {sum(lstm_times):.1f}s  |  Avg per epoch: {np.mean(lstm_times):.2f}s')

In [ ]:
print('Training Transformer...')
print('=' * 55)
tf_losses, tf_acc, tf_times = train_model(transformer_model, X, Y)
print(f'\nTotal: {sum(tf_times):.1f}s  |  Avg per epoch: {np.mean(tf_times):.2f}s')

In [ ]:
epochs = range(1, 16)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('LSTM vs Transformer — Training Comparison', fontsize=16, fontweight='bold', y=1.02)

LSTM_COLOR = '#E74C3C'
TRANSFORMER_COLOR = '#2471A3'

ax = axes[0]
ax.plot(epochs, lstm_losses, color=LSTM_COLOR, linewidth=2.5, marker='o', markersize=5, label='LSTM')
ax.plot(epochs, tf_losses, color=TRANSFORMER_COLOR, linewidth=2.5, marker='s', markersize=5, label='Transformer')
ax.set_title('Training Loss\n(lower is better)', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[1]
ax.plot(epochs, lstm_acc, color=LSTM_COLOR, linewidth=2.5, marker='o', markersize=5, label='LSTM')
ax.plot(epochs, tf_acc, color=TRANSFORMER_COLOR, linewidth=2.5, marker='s', markersize=5, label='Transformer')
ax.set_title('Accuracy\n(higher is better)', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[2]
ax.bar(['LSTM', 'Transformer'],
       [np.mean(lstm_times), np.mean(tf_times)],
       color=[LSTM_COLOR, TRANSFORMER_COLOR],
       width=0.5, edgecolor='white', linewidth=1.5)
ax.set_title('Avg Time Per Epoch\n(seconds)', fontsize=13, fontweight='bold')
ax.set_ylabel('Seconds')
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, v in enumerate([np.mean(lstm_times), np.mean(tf_times)]):
    ax.text(i, v + 0.005, f'{v:.3f}s', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('lstm_vs_transformer.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')

In [ ]:
print('=' * 55)
print('           FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  LSTM        — Final Accuracy: {lstm_acc[-1]:.1f}%   Avg Epoch: {np.mean(lstm_times):.3f}s')
print(f'  Transformer — Final Accuracy: {tf_acc[-1]:.1f}%   Avg Epoch: {np.mean(tf_times):.3f}s')
print('=' * 55)
faster = 'Transformer' if np.mean(tf_times) < np.mean(lstm_times) else 'LSTM'
ratio = max(np.mean(lstm_times), np.mean(tf_times)) / min(np.mean(lstm_times), np.mean(tf_times))
print(f'  {faster} was {ratio:.1f}x faster per epoch on this hardware')
print()
print('  Note: on a small CPU dataset the gap is modest.')
print('  At real GPU scale with long sequences, the Transformer')
print('  advantage compounds — GPU cores stay fully utilized')
print('  vs sitting idle waiting on each LSTM step.')
print('=' * 55)